# Environment Setup

Import required libraries and initialize the SageMaker environment.

In [1]:
# Import libraries and configure project settings

import warnings
warnings.filterwarnings("ignore")

import json
import tarfile
import boto3
import sagemaker

from sagemaker.session import Session

MODEL_PACKAGE_GROUP_NAME = "loan-approval-model-group"

PROJECT_PREFIX = "loanwise"

MODEL_FILE = "best_model.joblib"

sess = Session()

bucket = sess.default_bucket()

region = boto3.Session().region_name

role = sagemaker.get_execution_role()

print(f"SageMaker Version : {sagemaker.__version__}")
print(f"Region            : {region}")
print(f"Bucket            : {bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


SageMaker Version : 2.224.4
Region            : us-east-1
Bucket            : sagemaker-us-east-1-612256167011


# Load Best Model Artifact

Verify that the best model artifact generated in Notebook 2 is available.

In [2]:
# Verify model artifact exists

import os

if not os.path.exists(MODEL_FILE):
    raise FileNotFoundError(
        f"{MODEL_FILE} not found."
    )

print(f"Found model artifact: {MODEL_FILE}")

Found model artifact: best_model.joblib


# Create Deployment Package

Create a compressed model archive suitable for SageMaker registration.

In [3]:
# Create model.tar.gz

with tarfile.open(
    "model.tar.gz",
    "w:gz"
) as tar:

    tar.add(
        MODEL_FILE,
        arcname=MODEL_FILE
    )

print("model.tar.gz created successfully.")

model.tar.gz created successfully.


# Upload Model Package

Upload the deployment package to Amazon S3.

In [4]:
# Upload model package to S3

model_s3_uri = sess.upload_data(
    path="model.tar.gz",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/model-registry"
)

print(model_s3_uri)

s3://sagemaker-us-east-1-612256167011/loanwise/model-registry/model.tar.gz


# Create Model Metadata

Store metadata about the registered model.

In [5]:
# Create model metadata

model_metadata = {
    "project": "Loan Approval Prediction",
    "model_name": "best_model",
    "artifact": model_s3_uri,
    "framework": "Scikit-Learn/XGBoost",
    "status": "Approved"
}

with open(
    "model_metadata.json",
    "w"
) as f:

    json.dump(
        model_metadata,
        f,
        indent=4
    )

print("model_metadata.json created.")

model_metadata.json created.


# Create Model Package Group

Create a Model Package Group if it does not already exist.

In [6]:
# Create Model Package Group

sm_client = boto3.client("sagemaker")

try:

    sm_client.describe_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME
    )

    print(
        f"{MODEL_PACKAGE_GROUP_NAME} already exists."
    )

except Exception:

    sm_client.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
        ModelPackageGroupDescription=(
            "Loan Approval Prediction Models"
        )
    )

    print(
        f"{MODEL_PACKAGE_GROUP_NAME} created."
    )

loan-approval-model-group created.


# Register Model

Register the model package in SageMaker Model Registry.

In [7]:
# Register model package

response = sm_client.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    ModelApprovalStatus="Approved",
    InferenceSpecification={
        "Containers": [
            {
                "Image": (
                    "683313688378.dkr.ecr."
                    f"{region}.amazonaws.com/"
                    "sagemaker-scikit-learn:1.2-1-cpu-py3"
                ),
                "ModelDataUrl": model_s3_uri
            }
        ],
        "SupportedContentTypes": [
            "text/csv"
        ],
        "SupportedResponseMIMETypes": [
            "text/csv"
        ]
    }
)

model_package_arn = response[
    "ModelPackageArn"
]

print(model_package_arn)

arn:aws:sagemaker:us-east-1:612256167011:model-package/loan-approval-model-group/1


# Approve Model

Verify the model package approval status.

In [8]:
# Verify model package

package_details = sm_client.describe_model_package(
    ModelPackageName=model_package_arn
)

print(
    package_details["ModelApprovalStatus"]
)

Approved


# Upload Registry Metadata

Upload model metadata to Amazon S3.

In [9]:
# Upload metadata

metadata_s3_uri = sess.upload_data(
    path="model_metadata.json",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/registry"
)

print(metadata_s3_uri)

s3://sagemaker-us-east-1-612256167011/loanwise/registry/model_metadata.json


# Verify Registry Assets

Review generated Model Registry assets and metadata.

In [10]:
# Display registry artifacts

print("Model Package Group")
print("-------------------")
print(MODEL_PACKAGE_GROUP_NAME)

print("\nModel Package ARN")
print("-----------------")
print(model_package_arn)

print("\nModel Artifact")
print("--------------")
print(model_s3_uri)

print("\nMetadata")
print("--------")
print(metadata_s3_uri)

display(model_metadata)

Model Package Group
-------------------
loan-approval-model-group

Model Package ARN
-----------------
arn:aws:sagemaker:us-east-1:612256167011:model-package/loan-approval-model-group/1

Model Artifact
--------------
s3://sagemaker-us-east-1-612256167011/loanwise/model-registry/model.tar.gz

Metadata
--------
s3://sagemaker-us-east-1-612256167011/loanwise/registry/model_metadata.json


{'project': 'Loan Approval Prediction',
 'model_name': 'best_model',
 'artifact': 's3://sagemaker-us-east-1-612256167011/loanwise/model-registry/model.tar.gz',
 'framework': 'Scikit-Learn/XGBoost',
 'status': 'Approved'}

In [11]:
print(model_s3_uri)

s3://sagemaker-us-east-1-612256167011/loanwise/model-registry/model.tar.gz
